# SaCeSS vs. pyscat

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from lib import (
    OPTIMIZER_OVERVIEW_PATH,
    PROBLEM_OVERVIEW_PATH,
    RELATIVE_WALLTIME_LIMIT,
    get_threshold,
)

optimizer_df = pd.read_csv(OPTIMIZER_OVERVIEW_PATH)
problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)

# the pre-computed optimality gaps
df = pd.read_csv(
    "out/optimality_gaps.csv",
    usecols=[
        "problem",
        "optimizer",
        "run_idx",
        "fx_best",
        "site",
        "optimality_gap_site_with_pyscat",
    ],
)
df = (
    df.join(
        optimizer_df.set_index(["output_dir"]),
        on="optimizer",
        how="left",
        validate="many_to_one",
    )
    .query("excluded == False")
    .query("site == 'marvin'")
    .query("package == 'SaCeSS' or package == 'pyscat'")
)
assert (df.optimality_gap_site_with_pyscat >= 0).all(), df[
    df.optimality_gap_site_with_pyscat < 0
]

df_og_all_runs = df

# aggregate runs: best site-specific optimality gap
df_og_best_run = (
    df_og_all_runs.query("site == 'marvin'")
    .groupby(["problem", "optimizer"], as_index=False)
    .agg(
        optimality_gap=("optimality_gap_site_with_pyscat", "min"),
    )
    .join(
        optimizer_df.set_index(["output_dir"]),
        on="optimizer",
        how="left",
        validate="many_to_one",
    )
)
assert (df_og_best_run.optimality_gap >= 0).all(), df_og_best_run[
    df_og_best_run.optimality_gap < 0
]

df = df_og_best_run
df

In [ ]:
# Fig 7B
label_to_color = {
    "SaCeSS": "C0",
    "SaCeSS+Ipopt": "C1",
    "SaCeSS+DHC": "C2",
    "pySaCeSS": "C0",
    "pySaCeSS+cyipopt": "C1",
    "pySaCeSS+CMA-ES": "C3",
    "pySaCeSS+fides": "C4",
}


def plot_solved_problems_over_suboptimality(df):
    x_max = max(1e4, df.optimality_gap.max())

    assert set(label_to_color) == set(df.optimizer_label.unique()), set(
        label_to_color
    ) ^ set(df.optimizer_label.unique())
    order = list(label_to_color.keys())
    df["optimizer_label"] = pd.Categorical(
        df["optimizer_label"], categories=order, ordered=True
    )

    for i, (optimizer, grouped) in enumerate(
        df.groupby("optimizer_label", observed=False)
    ):
        grouped = grouped.sort_values("optimality_gap")
        x = grouped.optimality_gap.values
        if x[0] != 0:
            x = np.insert(x, 0, 0)  # add 0 at the beginning
        if x[-1] < x_max:
            x = np.append(x, x_max)
        # offset to avoid overplotting
        offset = -i * 0.05
        # number of solved problems; steps at each optimality gap value
        y = (
            np.array([(grouped.optimality_gap <= og).sum() for og in x])
            + offset
        )
        plt.step(
            x,
            y,
            where="post",
            label=optimizer,
            linestyle="--" if grouped.package.iloc[0] == "pyscat" else "-",
            color=label_to_color[optimizer],
        )
    plt.xscale("symlog", linthresh=1)
    ax = plt.gca()

    # generate minor tick positions across your x range
    minor_ticks = [0.1]
    for exp in range(-1, int(np.log10(x_max))):
        for sub in np.arange(2, 10):
            minor_ticks.append(sub * 10**exp)
    ax.set_xticks(minor_ticks, minor=True)
    ax.tick_params(axis="x", which="minor", length=2, color="black", width=0.8)

    plt.xlabel("Optimality gap threshold")
    plt.ylabel("# Solved problems")
    plt.legend()

    # plot thresholds
    for alpha in [0.05, 0.01, 0.001]:
        thresh = get_threshold(percentile=1 - alpha)
        plt.axvline(
            thresh,
            color="gray",
            linestyle="--",
            label=f"chi2 threshold (alpha={alpha})",
        )
        # label next to each line
        plt.text(
            thresh,
            plt.ylim()[1] * 0.02,
            f"$\\alpha={alpha}$, $OG={thresh:.2f}$",
            fontsize="small",
            rotation=90,
            verticalalignment="bottom",
            horizontalalignment="right",
        )


with plt.rc_context(
    {
        "figure.figsize": (17.5 / 2.54, 2.6),
        "figure.dpi": 300,
        "font.size": 7,
        "lines.markersize": 1,
        "lines.linewidth": 0.8,
        "figure.constrained_layout.use": True,
    }
):
    plot_solved_problems_over_suboptimality(df_og_best_run)
    plt.savefig("out/Figure7B.svg")
    plt.savefig("out/Figure7B.pdf")
    plt.show()

In [ ]:
# Fig 7C --- load FHT data
with open("../data/first_hitting_times_marvin.json") as f:
    data = json.load(f)

df_fht_all = pd.DataFrame(data)
del data

df_fht_all = df_fht_all.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
df_fht_all = df_fht_all.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)
df_fht_all = df_fht_all.explode(
    ["times", "thresholds", "significance_levels"]
).rename(
    columns={
        "significance_levels": "significance_level",
        "thresholds": "fval_threshold",
        "times": "first_hitting_time_s",
    }
)
for c in ["significance_level", "fval_threshold", "first_hitting_time_s"]:
    df_fht_all[c] = df_fht_all[c].astype(float)

df_fht_all = df_fht_all.query("excluded == False").query(
    "package == 'SaCeSS' or package == 'pyscat'"
)

assert df_fht_all.first_hitting_time_s.notna().all()
df_fht_all

In [ ]:
# Fig 7C
def plot_solved_problems_over_time(df_fht_all: pd.DataFrame):
    fig, axs = plt.subplots(
        1, 3, sharey=True, sharex=True, layout="constrained"
    )

    eff_time_limit_easy = RELATIVE_WALLTIME_LIMIT * 3 * 60 * 60
    eff_time_limit_hard = RELATIVE_WALLTIME_LIMIT * 9 * 60 * 60

    for i_ax, (ax, alpha) in enumerate(
        zip(
            axs,
            sorted(df_fht_all.significance_level.unique(), reverse=True),
            strict=False,
        )
    ):
        df = df_fht_all.query(f"significance_level == {alpha}")
        x_max = df.first_hitting_time_s[
            np.isfinite(df.first_hitting_time_s)
        ].max()

        assert set(label_to_color) == set(df.optimizer_label.unique()), set(
            label_to_color
        ) ^ set(df.optimizer_label.unique())

        order = list(label_to_color.keys())
        df["optimizer_label"] = pd.Categorical(
            df["optimizer_label"], categories=order, ordered=True
        )

        for i, (optimizer, grouped) in enumerate(
            df.groupby("optimizer_label", observed=False)
        ):
            grouped = grouped.sort_values("first_hitting_time_s")
            x = grouped.first_hitting_time_s.values
            if x[0] != 0:
                x = np.insert(x, 0, 0)
            if x[-1] < x_max:
                x = np.append(x, x_max)
            # offset to avoid overplotting
            offset = i * 0.05
            # number of solved problems; steps at each fht value
            y = (
                np.array(
                    [(grouped.first_hitting_time_s <= fht).sum() for fht in x]
                )
                + offset
            )
            ax.step(
                x,
                y,
                where="post",
                label=optimizer,
                linestyle="--" if grouped.package.iloc[0] == "pyscat" else "-",
                color=label_to_color[optimizer],
            )
        ax.set_xscale("log")
        ax.set_xlabel("Wall time [s]")

        if i_ax == len(axs) - 1:
            ax.legend(bbox_to_anchor=(1.05, 1))
        if i_ax == 0:
            ax.set_ylabel("# Solved problems")

        thresh = get_threshold(percentile=1 - alpha)
        ax.set_title(f"$\\alpha = {alpha}$, $OG \\leq {thresh:.2f}$")

        # plot walltime limits
        for time_limit, time_label in (
            (eff_time_limit_easy, "limit I"),
            (eff_time_limit_hard, "limit II"),
        ):
            ax.axvline(
                time_limit,
                color="grey",
                linestyle=(0, (0.01, 2)),
                label=f"{time_label} time limit",
                dash_capstyle="round",
            )
            ax.text(
                time_limit,
                ax.get_ylim()[1] * 0.05,
                time_label,
                rotation=90,
                verticalalignment="bottom",
                horizontalalignment="right",
                color="grey",
            )
        # millisecond range is irrelevant and distracting
        ax.set_xlim(left=1e-2)
        ax.set_ylim(0, df_fht_all.problem.nunique())


with plt.rc_context(
    {
        "figure.figsize": (17.5 / 2.54, 2.2),
        "figure.dpi": 300,
        "font.size": 7,
        "lines.markersize": 1,
        "lines.linewidth": 0.8,
        "figure.constrained_layout.use": True,
    }
):
    plot_solved_problems_over_time(df_fht_all)
    plt.savefig("out/Figure7C.svg")
    plt.show()

## Fig S5

In [ ]:
# Figs S5
def plot_optimizer_correlation(
    df: pd.DataFrame, opt1: str, opt2: str, ax: plt.Axes | None = None
):
    df_pivot = df.pivot(
        index="problem", columns="optimizer", values="optimality_gap"
    )
    x = df_pivot[opt1]
    y = df_pivot[opt2]
    assert np.all(x >= 0)
    assert np.all(y >= 0)
    lopt1 = optimizer_df.set_index("output_dir").loc[opt1, "optimizer_label"]
    lopt2 = optimizer_df.set_index("output_dir").loc[opt2, "optimizer_label"]

    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))

    for label, xx, yy in zip(df_pivot.index, x, y, strict=False):
        color = problem_df.set_index("short").loc[label, "problem_color"]
        ax.scatter(xx, yy, label=label, color=color)
    ax.set_xscale("symlog", linthresh=1e-4)
    ax.set_yscale("symlog", linthresh=1e-4)
    ax.set_xlabel(f"Optimality gap for {lopt1}")
    ax.set_ylabel(f"Optimality gap for {lopt2}")
    ax.set_title(
        f"Correlation of minimal optimality gaps between {lopt1} and {lopt2}"
    )

    alpha = 0.05
    thresh = get_threshold(1 - 0.05)
    ax.text(
        thresh,
        0,
        f"$\\alpha={alpha}$, $OG={thresh:.2f}$",
        fontsize="small",
        rotation=90,
        verticalalignment="bottom",
        horizontalalignment="right",
        color="lightgrey",
    )
    ax.text(
        0,
        thresh,
        f"$\\alpha={alpha}$, $OG={thresh:.2f}$",
        fontsize="small",
        verticalalignment="bottom",
        horizontalalignment="left",
        color="lightgrey",
    )
    ax.axhline(thresh, color="lightgrey", linestyle="--", zorder=0)
    ax.axvline(thresh, color="lightgrey", linestyle="--", zorder=0)

    ax.set_xlim(left=-5e-5)
    ax.set_ylim(bottom=-5e-5)

    ax.set_aspect("equal", adjustable="box")
    # diagonal
    low, high = ax.get_xlim()

    ax.plot(
        [low, high],
        [low, high],
        color="k",
        linestyle="-",
        linewidth=0.5,
        zorder=-100,
    )

    spear = spearmanr(x, y)
    ax.text(
        0.02,
        0.98,
        f"Spearman's $\\rho$ = {spear.correlation:.2f}\np = {spear.pvalue:.2e}",
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="left",
        # fontsize=8
    )
    ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0, ncol=2)


fig, axs = plt.subplots(
    nrows=2, ncols=1, figsize=(5, 10), sharex=True, sharey=True
)
fig.subplots_adjust(hspace=0.3)
plot_optimizer_correlation(df_og_best_run, "sacess", "pysacess", ax=axs[0])
plot_optimizer_correlation(
    df_og_best_run, "sacess_ipopt", "pysacess_ipopt", ax=axs[1]
)
axs[1].get_legend().remove()

axs[0].set_title(
    "A   " + axs[0].get_title(), fontweight="bold", loc="left", x=-0.25
)
axs[1].set_title(
    "B   " + axs[1].get_title(), fontweight="bold", loc="left", x=-0.25
)
axs[0].set_title("")
axs[1].set_title("")

plt.savefig("out/FigureS8_pysacess_og_correlation.pdf", bbox_inches="tight")
plt.show()